<a href="https://colab.research.google.com/github/Samuel22-ai/ANALYST_LAB_AFRICA_INTERNSHIP/blob/main/Second_Test_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.layers import Input, Cropping2D, Dropout, SeparableConv2D, BatchNormalization, ReLU, GlobalAveragePooling2D, Dense, Concatenate, Reshape, Conv2DTranspose, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam


# Forces CuDNN and GPU kernels to be mathematically deterministic
tf.config.experimental.enable_op_determinism()

# Sets Python, NumPy, and TF global seeds simultaneously
tf.keras.utils.set_random_seed(42)

# Verify GPU Acceleration
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


TensorFlow Version: 2.20.0
GPU Available: []


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("Loading binary matrices into Colab RAM...")
# Load the Training Data
X_imfcc_train = np.load('/content/drive/MyDrive/processed/acoustic_train_v6/X_acoustic_imfcc.npy')
X_ae_train = np.load('/content/drive/MyDrive/processed/acoustic_train_v6/X_acoustic_ae.npy')
y_train = np.load('/content/drive/MyDrive/processed/acoustic_train_v6/y_acoustic.npy')


X_imfcc_test = np.load('/content/drive/MyDrive/processed/acoustic_test_v6/X_acoustic_imfcc.npy')
X_ae_test = np.load('/content/drive/MyDrive/processed/acoustic_test_v6/X_acoustic_ae.npy')
y_test = np.load('/content/drive/MyDrive/processed/acoustic_test_v6/y_acoustic.npy')

mu_imfcc = np.mean(X_imfcc_train)
sigma_imfcc = np.std(X_imfcc_train) + 1e-8

X_imfcc_train_scaled = (X_imfcc_train - mu_imfcc) / sigma_imfcc
X_imfcc_test_scaled = (X_imfcc_test - mu_imfcc) / sigma_imfcc

print(f"Train Dataset Normalized. New Mean: {np.mean(X_imfcc_train_scaled):.2f}, StdDev: {np.std(X_imfcc_train_scaled):.2f}")
print(f"Test Dataset Normalized. New Mean: {np.mean(X_imfcc_test_scaled):.2f}, StdDev: {np.std(X_imfcc_test_scaled):.2f}")

# Prepare the data for Neural Network
X_imfcc_train_scaled = np.expand_dims(X_imfcc_train_scaled, axis=-1)
X_imfcc_test_scaled = np.expand_dims(X_imfcc_test_scaled, axis=-1)

np.save('/content/drive/MyDrive/processed/imfcc_mu.npy', mu_imfcc)
np.save('/content/drive/MyDrive/processed/imfcc_sigma.npy', sigma_imfcc)

print(f"Train Data Loaded. IMFCC Shape: {X_imfcc_train_scaled.shape}")
print(f"Test Data Loaded. IMFCC Shape: {X_imfcc_test_scaled.shape},")

Loading binary matrices into Colab RAM...
Train Dataset Normalized. New Mean: 0.00, StdDev: 1.00
Test Dataset Normalized. New Mean: 0.15, StdDev: 0.95
Train Data Loaded. IMFCC Shape: (112160, 32, 16, 1)
Test Data Loaded. IMFCC Shape: (51600, 32, 16, 1),


In [ ]:
mu_ae = np.mean(X_ae_train)
sigma_ae = np.std(X_ae_train) + 1e-8

X_ae_train_scaled = (X_ae_train - mu_ae) / sigma_ae
X_ae_test_scaled = (X_ae_test - mu_ae) / sigma_ae

print(f"Train Dataset Normalized (AE). New Mean: {np.mean(X_ae_train_scaled):.2f}, StdDev: {np.std(X_ae_train_scaled):.2f}")
print(f"Test Dataset Normalized (AE). New Mean: {np.mean(X_ae_test_scaled):.2f}, StdDev: {np.std(X_ae_test_scaled):.2f}")


np.save('/content/drive/MyDrive/processed/ae_mu.npy', mu_ae)
np.save('/content/drive/MyDrive/processed/ae_sigma.npy', sigma_ae)

print(f"Train Data Loaded. AE Shape: {X_ae_train_scaled.shape}")
print(f"Test Data Loaded. AE Shape: {X_ae_test_scaled.shape}")

Train Dataset Normalized (AE). New Mean: 0.00, StdDev: 1.00
Test Dataset Normalized (AE). New Mean: 0.05, StdDev: 1.11
Train Data Loaded. AE Shape: (112160, 6)
Test Data Loaded. AE Shape: (51600, 6)


In [ ]:
from tensorflow.keras.layers import Input, MaxPooling2D, LeakyReLU, Activation, Multiply, UpSampling2D, Conv2D, DepthwiseConv2D, SeparableConv2D, BatchNormalization, ReLU, Flatten, Dense, Reshape, Conv2DTranspose, Cropping2D, GlobalAveragePooling2D, Concatenate
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

def se_block(input_tensor, ratio=4, l2_penalty=1e-4):
    """
    Modern Lightweight Squeeze-and-Excitation Block for Edge AI.
    Uses decoupled activations to protect INT8 quantization constraints.
    """
    channels = input_tensor.shape[-1]

    # 1. Squeeze: Compress spatial dimensions to 1D
    se = GlobalAveragePooling2D()(input_tensor)

    # 2. Excite Bottleneck: Linear output, then strictly bounded ReLU
    se = Dense(channels // ratio, activation='linear', kernel_regularizer=l2(l2_penalty))(se)
    se = ReLU(max_value=6.0)(se) # Bounded to protect INT8 scale, prevents infinite spikes

    # Expansion back to original channel count
    # Sigmoid is safe here (TFLite Micro handles this via efficient lookup tables)
    se = Dense(channels, activation='sigmoid', kernel_regularizer=l2(l2_penalty))(se)

    # 3. Reshape and Multiply
    se = Reshape((1, 1, channels))(se)
    weighted_tensor = Multiply()([input_tensor, se])

    return weighted_tensor


def build_dual_stream_autoencoder_acoustic_model(l2_penalty=1e-4):
    """
    Constructs the hardware-aware, dual-input Functional Keras lightweight Autoencoder Model.
    With L2 weight decay to prevent overfitting on edge acoustic data.
    """
    spec_input = Input(shape=(32, 16, 1), name="spec_input")

    # --- THE ENCODER (Compressing to the Postage Stamp) ---
    # Layer 1: Temporal Edge Detection
    # Preserving Time Frames by using kernel_size: (1,3) to track time changes and Maxpooling: (2,1) to preserve time frame
    # Shape: (32, 16, 1) -> (16, 16, 16)
    x = Conv2D(filters=16, kernel_size=(1,3), padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(spec_input)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = se_block(x, ratio=4, l2_penalty=l2_penalty)
    x = MaxPooling2D(pool_size=(2,1))(x)

    # Layer 2: Extract receptive field to capture the physical impact envelope
    # Shape: (16, 16, 16) -> (8, 8, 32)
    x = DepthwiseConv2D(kernel_size=(3,3), padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = Conv2D(filters=32, kernel_size=(1, 1), padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = se_block(x, ratio=4, l2_penalty=l2_penalty)
    x = MaxPooling2D(pool_size=(2, 2))(x)

    # Layer 3
    # Shape: (8, 8, 32) -> (4, 4, 64)
    x = DepthwiseConv2D(kernel_size=(3,3), padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = Conv2D(filters=64, kernel_size=(1, 1), padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = se_block(x, ratio=4, l2_penalty=l2_penalty)
    x = MaxPooling2D(pool_size=(2, 2))(x)

    # Anti Bloat bottleneck
    # Shape: (4, 4, 64) -> (4, 4, 8)
    x = DepthwiseConv2D(kernel_size=(3,3), padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = Conv2D(filters=8, kernel_size=(1, 1), padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)

    # Shape: (4, 4, 8) -> (128, )
    x = Flatten(name="flattened_spec_activations")(x)
    # Shape: (128, ) -> (16,)
    x = Dense (16, activation="linear", kernel_regularizer=l2(l2_penalty), name="conv_activations")(x)
    x = BatchNormalization()(x)
    conv_activation = LeakyReLU(alpha=0.1, name="conv_activations_leaky")(x)

    # # Shape: (8, 4, 64) -> (64, )
    # x = GlobalAveragePooling2D()(x)
    # # Shape: (64, ) -> (32, )
    # conv_activation = Dense (32, activation="relu", kernel_regularizer=l2(l2_penalty), name="conv_activations")(x)

    bypass_input = Input(shape=(6,), name="bypass_input")
    x_bypass = Dense(16, activation="linear", kernel_regularizer=l2(l2_penalty))(bypass_input)
    x_bypass = BatchNormalization()(x_bypass)
    x_bypass = LeakyReLU(alpha=0.1, name="x_bypass_activations")(x_bypass)

    fused_acoustic = Concatenate(name="acoustic_fusion")([conv_activation, x_bypass])
    acoustic_signature = Dense(16, activation="linear", kernel_regularizer=l2(l2_penalty))(fused_acoustic)
    acoustic_signature = BatchNormalization()(acoustic_signature)
    acoustic_signature = LeakyReLU(alpha=0.1, name="final_acoustic_signature")(acoustic_signature)


    # --- THE DECODER (Reconstructing the Map) ---
    # First we rebuilt the spectrogram; the target shape is 4, 2, 8 from 32
    # Rebuild the flattened vector back into the 4x2x8 grid from 32
    dec_spec = Dense(4 * 4 * 8, activation="linear", name="dec_dense_spec")(acoustic_signature)
    dec_spec = BatchNormalization()(dec_spec)
    dec_spec = LeakyReLU(alpha=0.1, name="dec_dense_spec_activations")(dec_spec)

    dec_spec = Reshape((4, 4, 8), name="dec_spec_reshape")(dec_spec)

    # Shape (4,4,8) -> (4,4,64)
    dec_spec = DepthwiseConv2D(kernel_size=(3,3), padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(dec_spec)
    dec_spec = BatchNormalization()(dec_spec)
    dec_spec = ReLU(max_value=6.0)(dec_spec)
    dec_spec = Conv2D(filters=64, kernel_size=(1, 1), padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(dec_spec)
    dec_spec = BatchNormalization()(dec_spec)
    dec_spec = ReLU(max_value=6.0)(dec_spec)

    # Shape (4,4,64) -> (8,8,64)
    dec_spec = UpSampling2D(size=(2,2))(dec_spec)
    # Shape (8,8,64) -> (8,8,32)
    dec_spec = DepthwiseConv2D(kernel_size=(3,3), padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(dec_spec)
    dec_spec = BatchNormalization()(dec_spec)
    dec_spec = ReLU(max_value=6.0)(dec_spec)
    dec_spec = Conv2D(filters=32, kernel_size=(1, 1), padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(dec_spec)
    dec_spec = BatchNormalization()(dec_spec)
    dec_spec = ReLU(max_value=6.0)(dec_spec)

    # Shape (8,8,32) -> (16,16,32)
    dec_spec = UpSampling2D(size=(2,2))(dec_spec)
    # Shape (16,16,32) -> (16,16,16)
    dec_spec = DepthwiseConv2D(kernel_size=(3,3), padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(dec_spec)
    dec_spec = BatchNormalization()(dec_spec)
    dec_spec = ReLU(max_value=6.0)(dec_spec)
    dec_spec = Conv2D(filters=16, kernel_size=(1, 1), padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(dec_spec)
    dec_spec = BatchNormalization()(dec_spec)
    dec_spec = ReLU(max_value=6.0)(dec_spec)

    # Shape (16,16,16) -> (32,16,16)
    dec_spec = UpSampling2D(size=(2,1))(dec_spec)
    # Shape (32,16,16) -> (32,16,1)
    dec_spec = DepthwiseConv2D(kernel_size=(3,3), padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(dec_spec)
    dec_spec = BatchNormalization()(dec_spec)
    dec_spec = ReLU(max_value=6.0)(dec_spec)
    dec_spec = Conv2D(filters=1, kernel_size=(1, 1), padding="same", activation="linear", use_bias=True, kernel_regularizer=l2(l2_penalty))(dec_spec)

    # Shape (32,6,1) -> (512)
    dec_spec_flat = Flatten(name="Flattened_Spec_Output")(dec_spec)

    dec_bypass = Dense(6, activation="linear", name="dec_bypass_output")(acoustic_signature)

    # No concatenation. Output a list of multiple targets. Two distinct output streams
    autoencoder = Model(
        inputs={"spec_input": spec_input, "bypass_input": bypass_input},
        outputs={"Flattened_Spec_Output": dec_spec_flat, "dec_bypass_output": dec_bypass},
        name="Wide_Deep_Acoustic_Autoencoder"
    )
    encoder = Model(
        inputs={"spec_input": spec_input, "bypass_input": bypass_input},
        outputs={"final_acoustic_signature": acoustic_signature},
        name="Acoustic_Encoder_Only")

    return autoencoder, encoder

print("Initializing Unsupervised Autoencoder...")
autoencoder, encoder = build_dual_stream_autoencoder_acoustic_model()
autoencoder.load_weights('/content/drive/MyDrive/best_acoustic_autoencoder_new_v11.keras')
autoencoder.summary()
encoder.summary()

# Apply Loss Weights to balance the gradients (e.g., multiply bypass error by 25)
autoencoder.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss={'Flattened_Spec_Output': 'mae', 'dec_bypass_output': 'mse'},
    loss_weights={'Flattened_Spec_Output': 1.0, 'dec_bypass_output': 0.5} # Balances the 512 vs 6 discrepancy considering latent space expasion for bypass stats
)

Initializing Unsupervised Autoencoder...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "Wide_Deep_Acoustic_Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spec_input          │ (None, 32, 16, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 16,    │         48 │ spec_input[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 16,    │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 32, 16,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ re_lu[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4)         │         68 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 4)         │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         80 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 16)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 32, 16,    │          0 │ re_lu[0][0],      │
│                     │ 16)               │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ multiply[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (None, 16, 16,    │        144 │ max_pooling2d[0]… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │         64 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 16,    │        512 │ re_lu_2[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ re_lu_3[0][0]     │
│ (GlobalAveragePool… │                   │            │                 

 Total params: 18,667 (72.92 KB)

...

Model: "Acoustic_Encoder_Only"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spec_input          │ (None, 32, 16, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 16,    │         48 │ spec_input[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 16,    │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 32, 16,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ re_lu[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4)         │         68 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 4)         │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         80 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 16)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 32, 16,    │          0 │ re_lu[0][0],      │
│                     │ 16)               │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ multiply[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (None, 16, 16,    │        144 │ max_pooling2d[0]… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │         64 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 16,    │        512 │ re_lu_2[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ re_lu_3[0][0]     │
│ (GlobalAveragePool… │                   │            │                 

 Total params: 10,780 (42.11 KB)

 Trainable params: 10,220 (39.92 KB)

 Non-trainable params: 560 (2.19 KB)

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf


def tf_augment_spec (inputs, label):
  """
  NAtive TF Augmentation pipeline for (32,16,1) IMFCC spectrograms
  Applies noise and masaking sequentially with independent porbabilities
  Graph-safe Native TF augmentation.
  Uses pure tensor mathematics to avoid AutoGraph control flow errors,
  and returns a new dictionary to prevent state mutation.
  """
  image = inputs["spec_input"]

  # 1. White noise injection (with a 50% chance)
  # Cast boolean True/false into float 1.0/0.0
  apply_noise = tf.cast(tf.random.uniform([]) < 0.3, tf.float32)
  # Addition of a slight gaussian noise
  noise = tf.random.normal(shape=[32,16,1], mean=0.0, stddev=0.1)
  image = image + (apply_noise * noise) # Apply the noise only if the chance is met

  # 2. Frequency Masking (40% chance)
  apply_f = tf.cast(tf.random.uniform([]) < 0.2, tf.float32)
  # Set the maximum width of the mask (set as 6 out of 32), the actually width is dynamically random
  F = 6
  f = tf.random.uniform([], minval=1, maxval=F, dtype=tf.int32)
  f0 = tf.random.uniform([], minval=0, maxval=32 - f, dtype=tf.int32)

  # Create the boolean mask and zero out the selected freqency band (horizontal)
  indices = tf.range(32)
  mask_f = (indices >= f0) & (indices < f0 + f)
  mask_f = tf.cast(mask_f, tf.float32) # Converts the boolean value to 1.0 and 0.0
  mask_f = tf.reshape(mask_f, [32, 1, 1]) # Reshapes the 1D array to a 3D array with aligned shape

  # Multiple by (1 - mask) to zero out the band. The current array contain 0.0 for where we wish to keep and 1.0 for where we wish to zero out. We perform an addictive inverse
  image = image * (1.0 - (apply_f * mask_f))

  # 3. Time Masking (40% chance)
  apply_t = tf.cast(tf.random.uniform([]) < 0.2, tf.float32)
  # Set the maximum width of the max (set as 3 out of 16), the actual width is dynamically random
  T = 3
  t = tf.random.uniform([], minval=1, maxval=T, dtype=tf.int32)
  t0 = tf.random.uniform([], minval=0, maxval=16 - t, dtype=tf.int32)

  # Create the boolean mask and zero out the selected time band (vetical)
  indices = tf.range(16)
  mask_t = (indices >= t0) & (indices < t0 + t)
  mask_t = tf.cast(mask_t, tf.float32) # Converts the boolean value to 1.0 and 0.0
  mask_t = tf.reshape(mask_t, [1, 16, 1]) # Reshapes the 1D array to a 3D array with aligned shape

  # Multiply by addictive inverse of the mask
  image = image * (1.0 - (apply_t * mask_t))

  # Ensure the output stays strictly within a sensible range if the noise injection pushes it too far
  image = tf.clip_by_value(image, -3.0, 3.0)

  # Return band new dictionary of inputs rather than a mutation/inplace-modification
  return {
      "spec_input" : image,
      "bypass_input" : inputs["bypass_input"]
  }, label


X_imfcc_train, X_imfcc_val, X_bypass_train, X_bypass_val = train_test_split(
    X_imfcc_train_scaled,
    X_ae_train_scaled,
    test_size=0.2,
    random_state=42 # Ensures the random slice is identical every time you run it
)

X_imfcc_train_flat = X_imfcc_train.reshape(X_imfcc_train.shape[0], -1) # Flattens from (N, 32, 16, 1) -> (N, 512)
X_imfcc_val_flat = X_imfcc_val.reshape(X_imfcc_val.shape[0], -1)       # Flattens to (N, 512)


# Dictionary Mapped tf.data Pipeline
def create_dual_stream_dataset(imfcc, bypass, imfcc_flat):
  """
  Packages multi-modal arrays into Keras-compatible dictionaries.
  """
  # The keys for the dictionary must match the Input(name="...") exactly for both streams
  inputs = {
      "spec_input" : imfcc,
      "bypass_input" : bypass
  }

  # The keys must match the loss dictionary in model.compile exactly
  targets ={
      "Flattened_Spec_Output" : imfcc_flat,
      "dec_bypass_output" : bypass
  }

  # Bundle together and return a tuple of dictionaries (features, labels)
  return tf.data.Dataset.from_tensor_slices((inputs,targets))

print("Constructing optimized tf.data pipelines...")

# Build the training and validation pipeline
train_dataset = create_dual_stream_dataset(X_imfcc_train, X_bypass_train, X_imfcc_train_flat)
train_dataset = (
    train_dataset
    .shuffle(buffer_size=1024)
    .map(tf_augment_spec, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)
val_dataset = create_dual_stream_dataset(X_imfcc_val, X_bypass_val, X_imfcc_val_flat)
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

# train_dataset = tf.data.Dataset.from_tensor_slices((X_imfcc_train_scaled, X_imfcc_train_scaled))
# train_dataset = train_dataset.shuffle(X_imfcc_train_scaled.shape()[0]).batch(32).prefetch(tf.data.AUTOTUNE)



Constructing optimized tf.data pipelines...


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# 1. Early Stopping: Halt training if validation loss doesn't improve for 15 epochs
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    min_delta=1e-5,
    restore_best_weights=True,
    verbose=1
)

# 2. Reduce Learning Rate: Micro-adjust the optimizer step size as it gets closer to the minimum loss
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# 3. Model Checkpoint: Save the absolute best weights to disk during the run
checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/best_acoustic_autoencoder_new_v11.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Combine them into a list
callback_list = [early_stopping, reduce_lr, checkpoint]

# Start training with a high epoch count (the callbacks will stop it automatically)
print("Starting optimized training loop...")
history = autoencoder.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=150,
    callbacks=callback_list,
    verbose=1
)

Starting optimized training loop...
Epoch 1/150
2804/2804 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - Flattened_Spec_Output_loss: 0.3452 - dec_bypass_output_loss: 0.1456 - loss: 0.4440
Epoch 1: val_loss improved from None to 0.28209, saving model to /content/drive/MyDrive/best_acoustic_autoencoder_new_v11.keras

Epoch 1: finished saving model to /content/drive/MyDrive/best_acoustic_autoencoder_new_v11.keras
2804/2804 ━━━━━━━━━━━━━━━━━━━━ 127s 40ms/step - Flattened_Spec_Output_loss: 0.2950 - dec_bypass_output_loss: 0.0390 - loss: 0.3357 - val_Flattened_Spec_Output_loss: 0.2663 - val_dec_bypass_output_loss: 0.0041 - val_loss: 0.2821 - learning_rate: 0.0010
Epoch 2/150
2803/2804 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - Flattened_Spec_Output_loss: 0.2625 - dec_bypass_output_loss: 0.0046 - loss: 0.2766
Epoch 2: val_loss improved from 0.28209 to 0.26060, saving model to /content/drive/MyDrive/best_acoustic_autoencoder_new_v11.keras

Epoch 2: finished saving model to /content/drive/MyDrive/best_acoustic_a

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

print("Evaluating Acoustic Autoencoder on Test Set...")

# 1. Match the ground truth target shapes
# Flatten spectrogram target to (N, 512) to match 'Flattened_Spec_Output'
spec_target_flat = X_imfcc_test_scaled.reshape(X_imfcc_test_scaled.shape[0], -1)
bypass_target = X_ae_test_scaled

# 2. Predict reconstructions (using batch_size=64 to keep RAM usage low)
predictions = autoencoder.predict(
    {
        "spec_input": X_imfcc_test_scaled,   # Your 51,600 test spectrograms
        "bypass_input": X_ae_test_scaled # Your 51,600 test bypass stats
    }
)

# 2. Extract the physical arrays using the exact dictionary keys
pred_spec_flat = predictions["Flattened_Spec_Output"]
pred_bypass = predictions["dec_bypass_output"]

# 3. Frame-Level Error Calculation
# Spectrogram: MAE (matches the L1 training objective)
err_spec = np.mean(np.abs(spec_target_flat - pred_spec_flat), axis=1)

# Bypass: MSE (matches tabular training objective)
err_bypass = np.mean(np.square(bypass_target - pred_bypass), axis=1)

# Combined Anomaly Score (using your training loss weight ratio of 1.0 : 5.0)
total_frame_error = err_spec + (5.0 * err_bypass)

# 4. Aggregate to File-Level (40 frames = one 10-second recording)
frames_per_file = 40
num_files = len(y_test) // frames_per_file
usable_frames = num_files * frames_per_file

# Reshape into (num_files, 40) and average across each file's duration
spec_err_file = np.mean(err_spec[:usable_frames].reshape(num_files, frames_per_file), axis=1)
bypass_err_file = np.mean(err_bypass[:usable_frames].reshape(num_files, frames_per_file), axis=1)
total_err_file = np.mean(total_frame_error[:usable_frames].reshape(num_files, frames_per_file), axis=1)

# File-level ground truth (0 = Normal, 1 = Anomaly)
y_file_labels = y_test[:usable_frames].reshape(num_files, frames_per_file)[:, 0]

normal_mask = (y_file_labels == 0)
anomaly_mask = (y_file_labels == 1)

# 5. Separation Diagnostics
print("\n" + "="*55)
print("             STREAM RECONSTRUCTION SEPARATION           ")
print("="*55)
print(f"Spectrogram MAE:")
print(f"  - Normal Files:   {np.mean(spec_err_file[normal_mask]):.5f}")
print(f"  - Anomaly Files:  {np.mean(spec_err_file[anomaly_mask]):.5f}")
print(f"  - Separation Δ:   {np.mean(spec_err_file[anomaly_mask]) - np.mean(spec_err_file[normal_mask]):+.5f}")

print(f"\nBypass Stats MSE:")
print(f"  - Normal Files:   {np.mean(bypass_err_file[normal_mask]):.5f}")
print(f"  - Anomaly Files:  {np.mean(bypass_err_file[anomaly_mask]):.5f}")
print(f"  - Separation Δ:   {np.mean(bypass_err_file[anomaly_mask]) - np.mean(bypass_err_file[normal_mask]):+.5f}")

print(f"\nFused Combined Error:")
print(f"  - Normal Files:   {np.mean(total_err_file[normal_mask]):.5f}")
print(f"  - Anomaly Files:  {np.mean(total_err_file[anomaly_mask]):.5f}")

# 6. DCASE Benchmark Metrics
auc_spec = roc_auc_score(y_file_labels, spec_err_file)
auc_bypass = roc_auc_score(y_file_labels, bypass_err_file)
auc_fused = roc_auc_score(y_file_labels, total_err_file)

print("\n" + "="*55)
print("             AREA UNDER ROC CURVE (AUC)                ")
print("="*55)
print(f"Spectrogram Only AUC: {auc_spec * 100:.2f}%")
print(f"Bypass Stats Only AUC:{auc_bypass * 100:.2f}%")
print(f"Fused Stream AUC:     {auc_fused * 100:.2f}%")
print("="*55)

Evaluating Acoustic Autoencoder on Test Set...
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 22s 13ms/step

             STREAM RECONSTRUCTION SEPARATION           
Spectrogram MAE:
  - Normal Files:   0.22288
  - Anomaly Files:  0.24691
  - Separation Δ:   +0.02403

Bypass Stats MSE:
  - Normal Files:   0.00128
  - Anomaly Files:  0.00052
  - Separation Δ:   -0.00076

Fused Combined Error:
  - Normal Files:   0.22929
  - Anomaly Files:  0.24951

             AREA UNDER ROC CURVE (AUC)                
Spectrogram Only AUC: 75.69%
Bypass Stats Only AUC:51.08%
Fused Stream AUC:     74.35%


In [ ]:
# 2. Calculate the raw Absolute Error for every single pixel (Shape: N, 512)
pixel_errors = np.abs(spec_target_flat - pred_spec_flat)

# 3. Sort the errors for each file in ascending order
# (The highest errors are pushed to the far right of the array)
sorted_errors = np.sort(pixel_errors, axis=1)

# 4. Extract the "Hotspot": The 50 worst-reconstructed pixels (approx top 10%)
# This mathematically isolates the hammer strike / scrape
worst_50_pixels = sorted_errors[:, -50:]

# 5. Calculate the Hotspot Mean Error
hotspot_err = np.mean(worst_50_pixels, axis=1)

# 6. Calculate the new ROC-AUC based strictly on the Hotspot
hotspot_auc = roc_auc_score(y_test, hotspot_err)

print("=======================================================")
print("             HOTSPOT RECONSTRUCTION AUC                ")
print("=======================================================")
print(f"Top-50 Pixel Error AUC: {hotspot_auc * 100:.2f}%")
print("=======================================================")

             HOTSPOT RECONSTRUCTION AUC                
Top-50 Pixel Error AUC: 64.07%


In [ ]:
# 2. Extract Latent Vectors for the Training Set (Normal Data)
# Note: Use your un-augmented training dictionary here
train_latent = encoder.predict(
    {
        "spec_input": X_imfcc_train_scaled,
        "bypass_input": X_ae_train_scaled
    },
    batch_size=64
)
train_latents = train_latent["final_acoustic_signature"]

# 3. Calculate the "Normal Centroid" (The geometric center of your healthy pipeline)
normal_centroid = np.mean(train_latents, axis=0)

# 4. Extract Latent Vectors for the Test Set (Mixed Normal & Anomalies)
test_latent = encoder.predict(
    {
        "spec_input": X_imfcc_test_scaled,   # Your 51,600 test spectrograms
        "bypass_input": X_ae_test_scaled # Your 51,600 test bypass stats
    },
    batch_size=64
)
test_latents = test_latent["final_acoustic_signature"]

# 5. Calculate Euclidean Distance from the Test vectors to the Normal Centroid
# np.linalg.norm computes the exact physical distance in 32-dimensional space
latent_distances = np.linalg.norm(test_latents - normal_centroid, axis=1)

# 6. Calculate the new ROC-AUC based strictly on Latent Distance
latent_auc = roc_auc_score(y_test, latent_distances)

print("=======================================================")
print("             LATENT SPACE EUCLIDEAN AUC                ")
print("=======================================================")
print(f"Latent Distance AUC: {latent_auc * 100:.2f}%")
print("=======================================================")

1753/1753 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step
             LATENT SPACE EUCLIDEAN AUC                
Latent Distance AUC: 49.45%


In [ ]:
# from tensorflow.keras.layers import Input, MaxPooling2D, SeparableConv2D, BatchNormalization, ReLU, Flatten, Dense, Reshape, Conv2DTranspose, Cropping2D
# from tensorflow.keras.models import Model, load_model
# from tensorflow.keras.regularizers import l2
# from tensorflow.keras.optimizers import Adam

# autoencoder = load_model('/content/drive/MyDrive/best_acoustic_autoencoder_new_v4.keras')
# autoencoder.summary()

# print(autoencoder.output_names)

In [ ]:
# encoder = Model(
#     inputs=autoencoder.inputs,
#     outputs=autoencoder.get_layer("global_average_pooling2d").output
# )

In [ ]:
print("Executing Dual-Stream Inference...")

# 1. MULTI-MODAL PREDICTION
# Provide both inputs as a list to the dual-stream model
train_recon_spec, train_recon_bypass = autoencoder.predict(
    [X_imfcc_train_scaled, X_ae_train_scaled], batch_size=32
)
test_recon_spec, test_recon_bypass = autoencoder.predict(
    [X_imfcc_test_scaled, X_ae_test_scaled], batch_size=32
)

# 2. TARGET FLATTENING
# The decoder outputs a flattened 1D tensor (512 features). We must flatten the ground truth to match.
X_imfcc_train_flat = X_imfcc_train_scaled.reshape(X_imfcc_train_scaled.shape[0], -1)
X_imfcc_test_flat = X_imfcc_test_scaled.reshape(X_imfcc_test_scaled.shape[0], -1)

# 3. INDEPENDENT MSE CALCULATION
# Calculate mean error per sample (axis=1) for both the 512-D spatial and 6-D tabular streams
train_err_spec = np.mean(np.square(X_imfcc_train_flat - train_recon_spec), axis=1)
train_err_bypass = np.mean(np.square(X_ae_train_scaled - train_recon_bypass), axis=1)

test_err_spec = np.mean(np.square(X_imfcc_test_flat - test_recon_spec), axis=1)
test_err_bypass = np.mean(np.square(X_ae_test_scaled - test_recon_bypass), axis=1)

Executing Dual-Stream Inference...
3505/3505 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step


In [ ]:
y_test_pure = np.load('/content/drive/MyDrive/processed/acoustic_test_v5/y_acoustic.npy')

In [ ]:
from scipy import stats
from sklearn.preprocessing import StandardScaler

# 4. THE Z-SCORE SCALERS
# We fit standard scalers on the TRAINING errors so we know what "normal" error looks like
scaler_spec = StandardScaler().fit(train_err_spec.reshape(-1, 1))
scaler_bypass = StandardScaler().fit(train_err_bypass.reshape(-1, 1))

# Scale the test errors. Now, a score of 2.0 means "2 standard deviations higher than normal error"
test_z_spec = scaler_spec.transform(test_err_spec.reshape(-1, 1)).flatten()
test_z_bypass = scaler_bypass.transform(test_err_bypass.reshape(-1, 1)).flatten()

# 5. TRUE JOINT ANOMALY SCORE
# Because both are now scaled to Z-scores, we can safely add them together evenly.
# We apply np.maximum(0, array) to act as a ReLU for the Z-scores.
# This prevents negative (better-than-average) errors from masking positive anomalies.
clipped_z_spec = np.maximum(0, test_z_spec)
clipped_z_bypass = np.maximum(0, test_z_bypass)

test_recon_errors_pure = clipped_z_spec + clipped_z_bypass

# # 6. Temporal Pooling (Aggregating 250ms frames)
# errors_per_file_en = test_recon_errors_pure.reshape(1290, -1)
# test_recon_errors = np.percentile(errors_per_file_en, 98, axis=1)

# label_per_file = y_test_pure.reshape(1290, -1)
# y_test_modes = stats.mode(label_per_file, axis=1, keepdims=False)
# y_test = y_test_modes.mode if hasattr(y_test_modes, 'mode') else y_test_modes[0].squeeze()

In [ ]:
# 6. TEMPORAL VOTING (Sustained Attack Verification)
# --------------------------------------------------
errors_per_file_en = test_recon_errors_pure.reshape(1290, -1)

# Adjustable Parameters
FRAME_Z_THRESHOLD = 2.0        # A single 250ms frame needs a Z-score > 3.0 to trigger
REQUIRED_ANOMALY_RATIO = 0.2  # 25% of the frames in a file must trigger (e.g., 10 out of 40)

# Step A: Evaluate every frame individually (creates a Boolean array of True/False)
anomalous_frames = errors_per_file_en > FRAME_Z_THRESHOLD

# Step B: Calculate the percentage of triggered frames per file
# (np.mean on a boolean array treats True as 1.0 and False as 0.0, instantly giving the ratio)
anomaly_ratio_per_file = np.mean(anomalous_frames, axis=1)

# Step C: Final binary prediction for the entire file
file_predictions = (anomaly_ratio_per_file >= REQUIRED_ANOMALY_RATIO).astype(int)

# Extract ground truth labels per file
label_per_file = y_test_pure.reshape(1290, -1)
y_test_modes = stats.mode(label_per_file, axis=1, keepdims=False)
y_test = y_test_modes.mode if hasattr(y_test_modes, 'mode') else y_test_modes[0].squeeze()

# Calculate Accuracy/Metrics for the terminal
correct_predictions = np.sum(file_predictions == y_test)
total_files = len(y_test)
print(f"File-Level Accuracy: {(correct_predictions / total_files) * 100:.2f}%")




File-Level Accuracy: 58.29%


In [ ]:
# # 6. HISTOGRAM PLOTTING
# plt.figure(figsize=(10, 6))
# plt.hist(test_recon_errors[y_test==0], bins=50, alpha=0.6, color='blue', label='Normal Pipeline')
# plt.hist(test_recon_errors[y_test==1], bins=50, alpha=0.6, color='red', label='Vandalism/Anomaly Proxy')

# threshold_3 = np.percentile(test_recon_errors[y_test==0], 90) # Corrected to 95th to match your label
# threshold_4 = np.percentile(test_recon_errors[y_test==0], 99)

# plt.axvline(threshold_4, color='red', linestyle='dashed', linewidth=2,
#             label=f'Alarm Threshold (99th: {threshold_4:.2f})')
# plt.axvline(threshold_3, color='orange', linestyle='dashed', linewidth=2,
#             label=f'Warning Threshold (91st: {threshold_3:.2f})')

# plt.title("STM32H563 Joint Multi-Modal Autoencoder Separation using a Percentile of 90 per file", fontweight='bold')
# plt.xlabel("Joint Anomaly Score (Weighted MSE)")
# plt.ylabel("Number of Aggregated Events")
# plt.legend()
# plt.grid(True, linestyle='--', alpha=0.5)
# plt.savefig('/content/drive/MyDrive/Autoencoder_Separation_new_v5', dpi=300, bbox_inches='tight')
# plt.show()

# 7. HISTOGRAM PLOTTING (Plotting the Ratios)
# --------------------------------------------------
plt.figure(figsize=(10, 6))

# We now plot the distribution of the anomaly *ratios* rather than the raw MSE
plt.hist(anomaly_ratio_per_file[y_test==0], bins=20, alpha=0.6, color='blue', label='Normal Pipeline')
plt.hist(anomaly_ratio_per_file[y_test==1], bins=20, alpha=0.6, color='red', label='Vandalism/Anomaly Proxy')

# The threshold line is now fixed at your adjustable ratio requirement
plt.axvline(REQUIRED_ANOMALY_RATIO, color='red', linestyle='dashed', linewidth=2,
            label=f'Alarm Threshold ({REQUIRED_ANOMALY_RATIO * 100:.0f}% of frames)')

plt.title("STM32H563 File-Level Classification via Frame Voting Ratio", fontweight='bold')
plt.xlabel("Percentage of Anomalous Frames per File")
plt.ylabel("Number of Files")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('/content/drive/MyDrive/Autoencoder_Voting_Separation_v1', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Define the combinations you want to test
z_thresholds_to_test = [1.5 ,2.0, 2.5]
ratio_thresholds_to_test = [0.15, 0.20, 0.25]

# (Assuming 'errors_per_file_en' and 'y_test' are already loaded in memory)

# 2. Iterate through every combination
for z_thresh in z_thresholds_to_test:
    for ratio_thresh in ratio_thresholds_to_test:
        print(f"\n{'='*55}")
        print(f" EVALUATING: Frame Z-Threshold = {z_thresh} | Ratio = {ratio_thresh}")
        print(f"{'='*55}")

        # Step A: Apply Temporal Voting Logic
        anomalous_frames = errors_per_file_en > z_thresh
        anomaly_ratio_per_file = np.mean(anomalous_frames, axis=1)

        # Step B: Generate Predictions
        y_pred_au = (anomaly_ratio_per_file >= ratio_thresh).astype(int)

        # Step C: Print Text Metrics
        print("--- Classification Report ---")
        print(classification_report(y_test, y_pred_au, target_names=["Normal (0)", "Anomaly (1)"]))

        accuracy = accuracy_score(y_test, y_pred_au)
        print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

        # Step D: Plot and Dynamically Save Confusion Matrix
        cm = confusion_matrix(y_test, y_pred_au)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=["Normal", "Anomaly"],
                    yticklabels=["Normal", "Anomaly"])

        plt.ylabel('Actual (Ground Truth)')
        plt.xlabel('Predicted Label')
        plt.title(f'Confusion Matrix (Z={z_thresh}, Ratio={ratio_thresh})')

        # Format the filename dynamically so they save as distinct files
        save_name = f"/content/drive/MyDrive/CM_Z{z_thresh}_Ratio{ratio_thresh}.png"
        plt.savefig(save_name, dpi=300, bbox_inches='tight')

        # Display the plot in Colab, then close it to free up RAM
        plt.show()
        plt.close()

In [ ]:
# y_pred_au = (test_recon_errors > threshold_3).astype(int)

# print("--- Classification Report ---")
# print(classification_report(y_test, y_pred_au, target_names=["Normal (0)", "Anomaly (1)"]))


# accuracy = accuracy_score(y_test, y_pred_au)
# print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

# cm = confusion_matrix(y_test, y_pred_au)
# plt.figure(figsize=(6, 4))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
# plt.ylabel('Actual (Ground Truth)')
# plt.xlabel('Predicted Label')
# plt.title(f'Confusion Matrix (Threshold = {threshold_3})')
# plt.savefig("/content/drive/MyDrive/Confusion_Matrix_Autoencoder_Separation_new_v5", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
print("Loading binary matrices into Colab RAM...")
# Load the Training Data
X_env_train = np.load('/content/drive/MyDrive/processed/vibration_train/X_vibration_envelope.npy')
X_sband_train = np.load('/content/drive/MyDrive/processed/vibration_train/X_vibration_sub_band.npy')
y_vib_train = np.load('/content/drive/MyDrive/processed/vibration_train/y_vibration.npy')


X_env_test = np.load('/content/drive/MyDrive/processed/vibration_test/X_vibration_envelope.npy')
X_sband_test = np.load('/content/drive/MyDrive/processed/vibration_test/X_vibration_sub_band.npy')
y_vib_test = np.load('/content/drive/MyDrive/processed/vibration_test/y_vibration.npy')

mu_env = np.mean(X_env_train)
sigma_env = np.std(X_env_train) + 1e-8

X_env_train_scaled = (X_env_train - mu_env) / sigma_env
X_env_test_scaled = (X_env_test - mu_env) / sigma_env

print(f"Train Dataset Normalized. New Mean: {np.mean(X_env_train_scaled):.2f}, StdDev: {np.std(X_env_train_scaled):.2f}")
print(f"Test Dataset Normalized. New Mean: {np.mean(X_env_test_scaled):.2f}, StdDev: {np.std(X_env_test_scaled):.2f}")

# Prepare the data for Neural Network
X_env_train_scaled = np.expand_dims(X_env_train_scaled, axis=-1)
X_env_test_scaled = np.expand_dims(X_env_test_scaled, axis=-1)

np.save('/content/drive/MyDrive/processed/env_mu.npy', mu_env)
np.save('/content/drive/MyDrive/processed/env_sigma.npy', sigma_env)

print(f"Train Data Loaded. ENV Shape: {X_env_train_scaled.shape}")
print(f"Test Data Loaded. ENV Shape: {X_env_test_scaled.shape}")

Loading binary matrices into Colab RAM...
Train Dataset Normalized. New Mean: -0.00, StdDev: 1.00
Test Dataset Normalized. New Mean: -0.01, StdDev: 0.99
Train Data Loaded. ENV Shape: (112160, 100, 1)
Test Data Loaded. ENV Shape: (51600, 100, 1)


In [ ]:
mu_sband = np.mean(X_sband_train)
sigma_sband = np.std(X_sband_train) + 1e-8

X_sband_train_scaled = (X_sband_train - mu_sband) / sigma_sband
X_sband_test_scaled = (X_sband_test - mu_sband) / sigma_sband

print(f"Train Dataset Normalized (Sub-Band Statistics). New Mean: {np.mean(X_sband_train_scaled):.2f}, StdDev: {np.std(X_sband_train_scaled):.2f}")
print(f"Test Dataset Normalized (Sub-Band Statistics). New Mean: {np.mean(X_sband_test_scaled):.2f}, StdDev: {np.std(X_sband_test_scaled):.2f}")


np.save('/content/drive/MyDrive/processed/Sub_band_mu.npy', mu_sband)
np.save('/content/drive/MyDrive/processed/Sub_band_sigma.npy', sigma_sband)

print(f"Train Data Loaded. Sub_Band Shape: {X_sband_train_scaled.shape}")
print(f"Test Data Loaded. Sub_Band Shape: {X_sband_test_scaled.shape}")

Train Dataset Normalized (Sub-Band Statistics). New Mean: 0.00, StdDev: 1.00
Test Dataset Normalized (Sub-Band Statistics). New Mean: 0.01, StdDev: 1.03
Train Data Loaded. Sub_Band Shape: (112160, 16)
Test Data Loaded. Sub_Band Shape: (51600, 16)


In [ ]:
from tensorflow.keras.layers import Input, MaxPooling1D, UpSampling1D, Conv1D, DepthwiseConv1D, SeparableConv1D, BatchNormalization, ReLU, Flatten, Dense, Reshape, Conv1DTranspose, Cropping1D, GlobalAveragePooling1D, Concatenate
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

def build_dual_stream_autoencoder_vibration_model(l2_penalty=1e-4):
    """
    Constructs the hardware-aware, dual-input Functional Keras lightweight Autoencoder Model.
    With L2 weight decay to prevent overfitting on edge vibration data.
    """
    env_input = Input(shape=(100, 1), name="env_input")

    # --- THE ENCODER ---
    # Layer 1
    # Shape: (100, 1) -> (50, 16)
    x = Conv1D(filters=16, kernel_size=5, padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(env_input)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Layer 2
    # Shape: (50, 16) -> (25, 32)
    x = DepthwiseConv1D(kernel_size=5, padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = Conv1D(filters=32, kernel_size=1, padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Layer 3
    # Shape: (25, 32) -> (25, 64)
    x = DepthwiseConv1D(kernel_size=5, padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)
    x = Conv1D(filters=64, kernel_size=1, padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)

    # THE ANTI-BLOAT BOTTLENECK
    # Shape: (25, 64) -> (25, 16)
    x = Conv1D(filters=16, kernel_size=1, padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)

    # Shape: (25, 16) -> (400,)
    x = Flatten(name="flattened_cnn_activations")(x)

    # Shape: (400,) -> (16,)
    vib_feat = Dense(16, activation="relu", kernel_regularizer=l2(l2_penalty), name='vib_cnn_feats')(x)

    # Shape: (25, 32) -> (800,) -> (16,)
    # x = Flatten(name="flattened_cnn_activations")(x)
    # conv_activation = Dense (16, activation="relu", kernel_regularizer=l2(l2_penalty), name="conv_activations")(x)
    # Shape: (25, 64) -> (64,) -> (16,)
    # vib_cnn_activation = GlobalAveragePooling1D(name='vib_cnn_activations')(x)
    # vib_feat = Dense(16, activation="relu", kernel_regularizer=l2(l2_penalty), name='vib_cnn_feats')(vib_cnn_activation)

    sband_input = Input(shape=(16,), name="sub_band_input")
    v_sband = Dense(16, activation="relu", kernel_regularizer=l2(l2_penalty), name="x_sub_band_feats")(sband_input)

    fused_vibration = Concatenate(name="vibration_fusion")([vib_feat, v_sband])
    vibration_signature = Dense(32, activation="linear", kernel_regularizer=l2(l2_penalty))(fused_vibration)
    vibration_signature = BatchNormalization()(vibration_signature)
    vibration_signature = LeakyReLU(negative_slope=0.1, name="final_vibration_signature")(vibration_signature)


    # --- THE DECODER (Reconstructing the Map) ---
    # Rebuild the flattened vector back into the (25, 64) grid from 32
    dec_env = Dense(25 * 16, activation="relu", name="dec_env_dense")(vibration_signature)
    dec_env = Reshape((25, 16), name="dec_env_reshape")(dec_env)


    # Shape (25, 16) -> (25, 64)
    dec_env = DepthwiseConv1D(kernel_size=5, padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(dec_env)
    dec_env = BatchNormalization()(dec_env)
    dec_env = ReLU(max_value=6.0)(dec_env)
    dec_env = Conv1D(filters=64, kernel_size=1, padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(dec_env)
    dec_env = BatchNormalization()(dec_env)
    dec_env = ReLU(max_value=6.0)(dec_env)

    # Shape (25, 64) -> (25, 32)
    dec_env = DepthwiseConv1D(kernel_size=5, padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(dec_env)
    dec_env = BatchNormalization()(dec_env)
    dec_env = ReLU(max_value=6.0)(dec_env)
    dec_env = Conv1D(filters=32, kernel_size=1, padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(dec_env)
    dec_env = BatchNormalization()(dec_env)
    dec_env = ReLU(max_value=6.0)(dec_env)

    # Shape (25,32) -> (50, 32)
    dec_env = UpSampling1D(size=2)(dec_env)
    # Shape (50, 32) -> (50, 16)
    dec_env = DepthwiseConv1D(kernel_size=5, padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(dec_env)
    dec_env = BatchNormalization()(dec_env)
    dec_env = ReLU(max_value=6.0)(dec_env)
    dec_env = Conv1D(filters=16, kernel_size=1, padding="same", use_bias=False, kernel_regularizer=l2(l2_penalty))(dec_env)
    dec_env = BatchNormalization()(dec_env)
    dec_env = ReLU(max_value=6.0)(dec_env)

    # Shape (50, 16) -> (100, 16)
    dec_env = UpSampling1D(size=2)(dec_env)
    # Shape (100, 16) -> (100, 1)
    dec_env = DepthwiseConv1D(kernel_size=5, padding="same", use_bias=False, depthwise_regularizer=l2(l2_penalty))(dec_env)
    dec_env = BatchNormalization()(dec_env)
    dec_env = ReLU(max_value=6.0)(dec_env)
    dec_env = Conv1D(filters=1, kernel_size=1, padding="same", activation="linear", use_bias=True, kernel_regularizer=l2(l2_penalty))(dec_env)

    # Shape (100, 1) -> (100)
    dec_env_flat = Flatten(name="Flattened_Env_Output")(dec_env)

    dec_sband = Dense(16, activation="linear", name="dec_sband_output")(vibration_signature)

    # No concatenation. Output a list of multiple targets. Two distinct output streams
    autoencoder = Model(inputs=[env_input, sband_input], outputs=[dec_env_flat, dec_sband], name="Wide_Deep_Vibration_Autoencoder")
    encoder = Model(inputs=[env_input, sband_input], outputs=vibration_signature, name="Vibration_Encoder_Only")

    return autoencoder, encoder

print("Initializing Unsupervised Autoencoder...")
autoencoder, encoder = build_dual_stream_autoencoder_vibration_model()
autoencoder.summary()
encoder.summary()

# Apply Loss Weights to balance the gradients (e.g., multiply bypass error by 50)
autoencoder.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss={'Flattened_Env_Output': 'mse', 'dec_sband_output': 'mse'},
    loss_weights={'Flattened_Env_Output': 1.0, 'dec_sband_output': 6.25} # Slightly balances the 100 vs 16 difference despite latent space expansion of sband
)

Initializing Unsupervised Autoencoder...


Model: "Wide_Deep_Vibration_Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ env_input           │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 100, 16)   │         80 │ env_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 16)   │         64 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_14 (ReLU)     │ (None, 100, 16)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 50, 16)    │          0 │ re_lu_14[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d    │ (None, 50, 16)    │         80 │ max_pooling1d[0]… │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 16)    │         64 │ depthwise_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_15 (ReLU)     │ (None, 50, 16)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 32)    │        512 │ re_lu_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 32)    │        128 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_16 (ReLU)     │ (None, 50, 32)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 25, 32)    │          0 │ re_lu_16[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d_1  │ (None, 25, 32)    │        160 │ max_pooling1d_1[… │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 32)    │        128 │ depthwise_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_17 (ReLU)     │ (None, 25, 32)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 25, 64)    │      2,048 │ re_lu_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 64)    │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_18 (ReLU)     │ (None, 25, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 25, 16)    │      1,024 │ re_lu_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 16)    │         64 │ conv1d_3[0][0]  

 Total params: 31,409 (122.69 KB)

 Trainable params: 30,513 (119.19 KB)

...

Model: "Vibration_Encoder_Only"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ env_input           │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 100, 16)   │         80 │ env_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 16)   │         64 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_14 (ReLU)     │ (None, 100, 16)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 50, 16)    │          0 │ re_lu_14[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d    │ (None, 50, 16)    │         80 │ max_pooling1d[0]… │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 16)    │         64 │ depthwise_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_15 (ReLU)     │ (None, 50, 16)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 32)    │        512 │ re_lu_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 32)    │        128 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_16 (ReLU)     │ (None, 50, 32)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 25, 32)    │          0 │ re_lu_16[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d_1  │ (None, 25, 32)    │        160 │ max_pooling1d_1[… │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 32)    │        128 │ depthwise_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_17 (ReLU)     │ (None, 25, 32)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 25, 64)    │      2,048 │ re_lu_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 64)    │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_18 (ReLU)     │ (None, 25, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 25, 16)    │      1,024 │ re_lu_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 16)    │         64 │ conv1d_3[0][0]  

 Total params: 12,480 (48.75 KB)

 Trainable params: 12,064 (47.12 KB)

 Non-trainable params: 416 (1.62 KB)

In [ ]:
from sklearn.model_selection import train_test_split


X_env_train, X_env_val, X_sband_train, X_sband_val = train_test_split(
    X_env_train_scaled,
    X_sband_train_scaled,
    test_size=0.2,
    random_state=42 # Ensures the random slice is identical every time you run it
)

X_env_train_flat = X_env_train.reshape(X_env_train.shape[0], -1) # Flattens from (N, 100, 1) -> (N, 100)
X_env_val_flat = X_env_val.reshape(X_env_val.shape[0], -1)       # Flattens to (N, 100)


# Dictionary Mapped tf.data Pipeline
def create_dual_stream_dataset(env, sband, env_flat):
  """
  Packages multi-modal arrays into Keras-compatible dictionaries.
  """
  # The keys for the dictionary must match the Input(name="...") exactly for both streams
  inputs = {
      "env_input" : env,
      "sub_band_input" : sband
  }

  # The keys must match the loss dictionary in model.compile exactly
  targets ={
      "Flattened_Env_Output" : env_flat,
      "dec_sband_output" : sband
  }

  # Bundle together and return a tuple of dictionaries (features, labels)
  return tf.data.Dataset.from_tensor_slices((inputs,targets))

print("Constructing optimized tf.data pipelines...")

# Build the training and validation pipeline
train_dataset = create_dual_stream_dataset(X_env_train, X_sband_train, X_env_train_flat)
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = create_dual_stream_dataset(X_env_val, X_sband_val, X_env_val_flat)
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)




Constructing optimized tf.data pipelines...


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# 1. Early Stopping: Halt training if validation loss doesn't improve for 15 epochs
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    min_delta=1e-5,
    restore_best_weights=True,
    verbose=1
)

# 2. Reduce Learning Rate: Micro-adjust the optimizer step size as it gets closer to the minimum loss
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# 3. Model Checkpoint: Save the absolute best weights to disk during the run
checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/best_vibration_autoencoder_v1.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Combine them into a list
callback_list = [early_stopping, reduce_lr, checkpoint]

# Start training with a high epoch count (the callbacks will stop it automatically)
print("Starting optimized training loop...")
history = autoencoder.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=150,
    callbacks=callback_list,
    verbose=1
)

Starting optimized training loop...
Epoch 1/150
2799/2804 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - Flattened_Env_Output_loss: 0.7982 - dec_sband_output_loss: 0.2144 - loss: 2.1651
Epoch 1: val_loss improved from None to 0.73391, saving model to /content/drive/MyDrive/best_vibration_autoencoder_v1.keras

Epoch 1: finished saving model to /content/drive/MyDrive/best_vibration_autoencoder_v1.keras
2804/2804 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - Flattened_Env_Output_loss: 0.6472 - dec_sband_output_loss: 0.0796 - loss: 1.1711 - val_Flattened_Env_Output_loss: 0.5313 - val_dec_sband_output_loss: 0.0284 - val_loss: 0.7339 - learning_rate: 0.0010
Epoch 2/150
2803/2804 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - Flattened_Env_Output_loss: 0.5349 - dec_sband_output_loss: 0.0194 - loss: 0.6802
Epoch 2: val_loss improved from 0.73391 to 0.67510, saving model to /content/drive/MyDrive/best_vibration_autoencoder_v1.keras

Epoch 2: finished saving model to /content/drive/MyDrive/best_vibration_autoencoder_v1.keras
280

In [ ]:
print("Extracting Raw Feature Comparisons...")

# 1. Isolate the indices for Normal (0) and Anomaly (1) from your sliced/balanced labels
y_test_files = y_test.reshape(1290, 40)[:, 0]
normal_file_idx = np.where(y_test_files == 0)[0][0]  # Grab the very first Normal file
anomaly_file_idx = np.where(y_test_files == 1)[0][0] # Grab the very first Anomaly file

# Grab a frame right in the middle of the file (e.g., frame 20 out of 40)
normal_frame_idx = (normal_file_idx * 40) + 20
anomaly_frame_idx = (anomaly_file_idx * 40) + 20

# 2. Extract the Spectrograms (Removing the batch and channel dimensions for plotting)
spec_normal = X_imfcc_test[normal_frame_idx].reshape(32, 16)
spec_anomaly = X_imfcc_test[anomaly_frame_idx].reshape(32, 16)

# # 3. Extract the Envelopes (Removing the batch and channel dimensions)
# env_normal = X_env_test[normal_frame_idx].reshape(100)
# env_anomaly = X_env_test[anomaly_frame_idx].reshape(100)

# 4. Plotting the Physical Data
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Physical Feature Verification: Normal Pipeline vs Vandalism", fontsize=16, fontweight='bold')

# --- TOP ROW: INVERTED MEL-SPECTROGRAMS ---
ax1 = axes[0, 0]
cax1 = ax1.imshow(spec_normal, aspect='auto', origin='lower', cmap='viridis')
ax1.set_title("Normal Pipeline: IMFCC Spectrogram")
ax1.set_ylabel("Frequency Bins (Inverted)")
ax1.set_xlabel("Time Frames")
fig.colorbar(cax1, ax=ax1)

ax2 = axes[0, 1]
cax2 = ax2.imshow(spec_anomaly, aspect='auto', origin='lower', cmap='magma') # Different color map to highlight intensity
ax2.set_title("Vandalism Strike: IMFCC Spectrogram")
ax2.set_xlabel("Time Frames")
fig.colorbar(cax2, ax=ax2)

# # --- BOTTOM ROW: KINEMATIC ENVELOPES ---
# ax3 = axes[1, 0]
# ax3.plot(env_normal, color='blue', linewidth=2)
# ax3.set_title("Normal Pipeline: Kinematic Envelope")
# ax3.set_ylabel("Amplitude")
# ax3.set_xlabel("Time Steps (100)")
# ax3.grid(True, linestyle='--', alpha=0.6)
# ax3.set_ylim([0, 1.0]) # Assuming data is MinMax scaled between 0 and 1

# ax4 = axes[1, 1]
# ax4.plot(env_anomaly, color='red', linewidth=2)
# ax4.set_title("Vandalism Strike: Kinematic Envelope")
# ax4.set_xlabel("Time Steps (100)")
# ax4.grid(True, linestyle='--', alpha=0.6)
# ax4.set_ylim([0, 1.0])

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
from tensorflow.keras.layers import Input, MaxPooling2D, SeparableConv2D, BatchNormalization, ReLU, Flatten, Dense, Reshape, Conv2DTranspose, Cropping2D
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

autoencoder_acoustic, encoder_acoustic = build_dual_stream_autoencoder_acoustic_model()

# Load the trained weights into the FULL autoencoder
autoencoder_acoustic.load_weights("/content/drive/MyDrive/best_acoustic_autoencoder_new_v7.keras")


# autoencoder_acoustic = load_model('/content/drive/MyDrive/best_acoustic_autoencoder_new_v7.keras')
# autoencoder_acoustic.summary()

# print(autoencoder_acoustic.output_names)

In [ ]:
# The 'encoder' variable now automatically contains the fully trained weights!
# You can freeze it and use it immediately, or save it as a standalone file.
# encoder_acoustic.trainable = False
encoder_acoustic.save("/content/drive/MyDrive/best_acoustic_encoder.keras")
encoder_acoustic.summary()

# encoder_acoustic = Model(
#     inputs=autoencoder_acoustic.inputs,
#     outputs=autoencoder_acoustic.get_layer("global_average_pooling2d").output
# )

Model: "Acoustic_Encoder_Only"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spec_input          │ (None, 32, 16, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 32, 16,    │        144 │ spec_input[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 16,    │         64 │ conv2d_8[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_27 (ReLU)     │ (None, 32, 16,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 16, 8, 16) │          0 │ re_lu_27[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_7  │ (None, 16, 8, 16) │        144 │ max_pooling2d_3[… │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 8, 16) │         64 │ depthwise_conv2d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_28 (ReLU)     │ (None, 16, 8, 16) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 16, 8, 32) │        512 │ re_lu_28[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 8, 32) │        128 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_29 (ReLU)     │ (None, 16, 8, 32) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 8, 4, 32)  │          0 │ re_lu_29[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_8  │ (None, 8, 4, 32)  │        288 │ max_pooling2d_4[… │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 4, 32)  │        128 │ depthwise_conv2d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_30 (ReLU)     │ (None, 8, 4, 32)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 8, 4, 64)  │      2,048 │ re_lu_30[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 4, 64)  │        256 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_31 (ReLU)     │ (None, 8, 4, 64)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 4, 2, 64)  │          0 │ re_lu_31[0][0]  

 Total params: 9,664 (37.75 KB)

 Trainable params: 9,136 (35.69 KB)

 Non-trainable params: 528 (2.06 KB)

In [ ]:
autoencoder_vibration, encoder_vibration = build_dual_stream_autoencoder_vibration_model()

# Load the trained weights into the FULL autoencoder
autoencoder_vibration.load_weights("/content/drive/MyDrive/best_vibration_autoencoder_v1.keras")

# autoencoder_vibration = load_model('/content/drive/MyDrive/best_acoustic_autoencoder_v1.keras')
# autoencoder_vibration.summary()

# print(autoencoder_vibration.output_names)

In [ ]:
# The 'encoder' variable now automatically contains the fully trained weights!
# You can freeze it and use it immediately, or save it as a standalone file.
# encoder_vibration.trainable = False
encoder_vibration.save("/content/drive/MyDrive/best_vibration_encoder.keras")
encoder_vibration.summary()

# encoder_vibration = Model(
#     inputs=autoencoder_vibration.inputs,
#     outputs=autoencoder_vibration.get_layer("global_average_pooling2d").output
# )

Model: "Vibration_Encoder_Only"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ env_input           │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 100, 16)   │         80 │ env_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 16)   │         64 │ conv1d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_41 (ReLU)     │ (None, 100, 16)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 50, 16)    │          0 │ re_lu_41[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d_6  │ (None, 50, 16)    │         80 │ max_pooling1d_2[… │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 16)    │         64 │ depthwise_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_42 (ReLU)     │ (None, 50, 16)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 50, 32)    │        512 │ re_lu_42[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 32)    │        128 │ conv1d_9[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_43 (ReLU)     │ (None, 50, 32)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 25, 32)    │          0 │ re_lu_43[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d_7  │ (None, 25, 32)    │        160 │ max_pooling1d_3[… │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 32)    │        128 │ depthwise_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_44 (ReLU)     │ (None, 25, 32)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 25, 64)    │      2,048 │ re_lu_44[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 64)    │        256 │ conv1d_10[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_45 (ReLU)     │ (None, 25, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 25, 16)    │      1,024 │ re_lu_45[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 16)    │         64 │ conv1d_11[0][0] 

 Total params: 12,480 (48.75 KB)

 Trainable params: 12,064 (47.12 KB)

 Non-trainable params: 416 (1.62 KB)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Concatenate, ReLU, BatchNormalization, Dropout, LeakyReLU
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

def build_end_to_end_latent_autoencoder(trained_acoustic_encoder, trained_vibration_encoder, l2_penalty=1e-4):
    """
    Constructs the final Autoencoder.
    Takes raw arrays as input, processes them through frozen Stage 1 encoders,
    and trains a joint bottleneck to reconstruct the latent signatures.
    """
    # FREEZE STAGE 1 ENCODERS
    trained_acoustic_encoder.trainable = False
    trained_vibration_encoder.trainable = False

    # RAW INPUT DEFINITIONS
    spec_input = Input(shape=(32, 16, 1), name="spec_input")
    bypass_input = Input(shape=(6,), name="bypass_input")

    env_input = Input(shape=(100, 1), name="env_input")
    sband_input = Input(shape=(16,), name="sub_band_input")

    # EXTRACT SIGNATURES (Forward pass through frozen layers)
    # The models are called as layers, outputting the 32-value tensors
    acoustic_sig = trained_acoustic_encoder([spec_input, bypass_input])
    vibration_sig = trained_vibration_encoder([env_input, sband_input])

    # JOINT FUSION
    joint_fusion = Concatenate(name="latent_fusion")([acoustic_sig, vibration_sig])
    x = BatchNormalization()(joint_fusion)

    # Modality relationship extraction
    x = Dense(32, kernel_regularizer=l2(l2_penalty), kernel_initializer='he_normal', use_bias=False,  name='fusion_mixing_layer')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.1)(x)
    x = Dropout(0.1)(x)

    # THE TRAINABLE BOTTLENECK (Compressing to 8 neurons)
    joint_bottleneck = Dense(8, kernel_regularizer=l2(l2_penalty), kernel_initializer='he_normal', use_bias=False, name="joint_bottleneck")(x)
    joint_bottleneck = BatchNormalization()(joint_bottleneck)
    joint_bottleneck = LeakyReLU(alpha=0.1)(joint_bottleneck)

    # Decompression
    joint_bottleneck = Dense(64, use_bias=False, kernel_initializer='he_normal', kernel_regularizer=l2(l2_penalty))(joint_bottleneck)
    joint_bottleneck = BatchNormalization()(joint_bottleneck)
    joint_bottleneck = LeakyReLU(alpha=0.1)(joint_bottleneck)

    # RECONSTRUCTION
    # Target 1: The Spectrogram (32x16 = 512 flat pixels)
    dec_spec = Dense(512, use_bias=True, activation="linear", kernel_initializer='he_normal', name="dense_spec")(joint_bottleneck)
    out_spec = Reshape((32, 16, 1), name="out_spec")(dec_spec)

    # Target 2: The Acoustic Stats
    out_bypass = Dense(6, use_bias=True, activation="linear", kernel_initializer='he_normal', name="out_bypass")(joint_bottleneck)

    # Target 3: The Kinematic Envelope (100 steps)
    dec_env = Dense(100, use_bias=True, activation="linear", kernel_initializer='he_normal', name="dense_env")(joint_bottleneck)
    out_env = Reshape((100, 1), name="out_env")(dec_env)

    # Target 4: The Vibration Stats
    out_sband = Dense(16, use_bias=True, activation="linear", kernel_initializer='he_normal', name="out_sband")(joint_bottleneck)

    final_autoencoder = Model(
        inputs=[spec_input, bypass_input, env_input, sband_input],
        outputs=[out_spec, out_bypass, out_env, out_sband],
        name="End_to_End_Latent_Autoencoder"
    )



    return final_autoencoder

autoencoder = build_end_to_end_latent_autoencoder(encoder_acoustic, encoder_vibration)
# autoencoder.load_weights("/content/drive/MyDrive/best_joint_autoencoder_v4.keras")

autoencoder.compile(
        optimizer=Adam(learning_rate=5e-4),
        loss={
            'out_spec': 'mse',
            'out_bypass': 'mse',
            'out_env' : 'mse',
            'out_sband' : 'mse'
        },
        loss_weights={
            'out_spec': 1.0,
            'out_bypass': 0.1,
            'out_env' : 1.0,
            'out_sband' : 0.1
        }
)

autoencoder.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "End_to_End_Latent_Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spec_input          │ (None, 32, 16, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bypass_input        │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ env_input           │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sub_band_input      │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Acoustic_Encoder_O… │ (None, 32)        │      9,664 │ spec_input[0][0], │
│ (Functional)        │                   │            │ bypass_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Vibration_Encoder_… │ (None, 32)        │     12,480 │ env_input[0][0],  │
│ (Functional)        │                   │            │ sub_band_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latent_fusion       │ (None, 64)        │          0 │ Acoustic_Encoder… │
│ (Concatenate)       │                   │            │ Vibration_Encode… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ latent_fusion[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_mixing_layer │ (None, 32)        │      2,048 │ batch_normalizat… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ fusion_mixing_la… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 32)        │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ leaky_re_lu[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ joint_bottleneck    │ (None, 8)         │        256 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8)         │         32 │ joint_bottleneck… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 8)         │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │        512 │ leaky_re_lu_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_4[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 66,842 (261.10 KB)

 Trainable params: 44,362 (173.29 KB)

 Non-trainable params: 22,480 (87.81 KB)

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting the entire dataset for the two stream - four inputs
(
    X_imfcc_train, X_imfcc_val,
    X_ae_train, X_ae_val,
    X_env_train, X_env_val,
    X_sband_train, X_sband_val
)= train_test_split(
    X_imfcc_train_scaled,
    X_ae_train_scaled,
    X_env_train_scaled,
    X_sband_train_scaled,
    test_size=0.2,
    random_state=42 # Ensures the random slice is identical every time you run it
)

print("The splitting has been completed")

The splitting has been completed


In [ ]:
# MULTI-MODAL PREDICTION
# Provide both inputs as a list to the dual-stream model
acoustic_train_signature = encoder_acoustic.predict(
    [X_imfcc_train, X_ae_train], batch_size=32
)

vibration_train_signature = encoder_vibration.predict(
    [X_env_train, X_sband_train], batch_size=32
)


2804/2804 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step
2804/2804 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step


In [ ]:
if acoustic_train_signature.shape == vibration_train_signature.shape:
  print(f"The shape of the vibration and acoustic signature is aligned")

The shape of the vibration and acoustic signature is aligned


In [ ]:
# MULTI-MODAL PREDICTION
# Provide both inputs as a list to the dual-stream model
acoustic_val_signature = encoder_acoustic.predict(
    [X_imfcc_val, X_ae_val], batch_size=32
)

vibration_val_signature = encoder_vibration.predict(
    [X_env_val, X_sband_val], batch_size=32
)

if acoustic_val_signature.shape == vibration_val_signature.shape:
  print(f"The shape of the vibration and acoustic signature is aligned")


701/701 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
701/701 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
The shape of the vibration and acoustic signature is aligned


In [ ]:

# Dictionary Mapped tf.data Pipeline
def create_end_to_end_dataset(spec, bypass, env, sband):
  """
  Packages multi-modal arrays into Keras-compatible dictionaries.
  """

  # The keys for the dictionary must match the Input(name="...") exactly for both streams
  inputs = {
      "spec_input" : spec,
      "bypass_input" : bypass,
      "env_input" : env,
      "sub_band_input" : sband
  }

  # The keys must match the loss dictionary in model.compile exactly
  targets ={
      "out_spec": spec,      # Reconstructing raw spectrogram
      "out_bypass": bypass,  # Reconstructing raw stats
      "out_env": env,        # Reconstructing raw envelope
      "out_sband": sband     # Reconstructing raw stats
  }

  # Bundle together and return a tuple of dictionaries (features, labels)
  return tf.data.Dataset.from_tensor_slices((inputs,targets))

print("Constructing optimized tf.data pipelines...")

# Build the training and validation pipeline
train_dataset = create_end_to_end_dataset(
    spec=X_imfcc_train,
    bypass=X_ae_train,
    env=X_env_train,
    sband=X_sband_train
)

train_dataset = train_dataset.shuffle(buffer_size=1024).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = create_end_to_end_dataset(
    spec=X_imfcc_val,
    bypass=X_ae_val,
    env=X_env_val,
    sband=X_sband_val
)

val_dataset = val_dataset.shuffle(buffer_size=1024).batch(32).prefetch(tf.data.AUTOTUNE)



Constructing optimized tf.data pipelines...


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# 1. Early Stopping: Halt training if validation loss doesn't improve for 15 epochs
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    min_delta=1e-5,
    restore_best_weights=True,
    verbose=1
)

# 2. Reduce Learning Rate: Micro-adjust the optimizer step size as it gets closer to the minimum loss
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# 3. Model Checkpoint: Save the absolute best weights to disk during the run
checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/best_joint_autoencoder_v5.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Combine them into a list
callback_list = [early_stopping, reduce_lr, checkpoint]

# Start training with a high epoch count (the callbacks will stop it automatically)
print("Starting optimized training loop...")
history = autoencoder.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=callback_list,
    verbose=1
)
print("Training complete")

Starting optimized training loop...
Epoch 1/100
2796/2804 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.3629 - out_bypass_loss: 0.1792 - out_env_loss: 0.8439 - out_sband_loss: 0.4762 - out_spec_loss: 0.4339
Epoch 1: val_loss improved from None to 0.82155, saving model to /content/drive/MyDrive/best_joint_autoencoder_v5.keras

Epoch 1: finished saving model to /content/drive/MyDrive/best_joint_autoencoder_v5.keras
2804/2804 ━━━━━━━━━━━━━━━━━━━━ 31s 8ms/step - loss: 1.0302 - out_bypass_loss: 0.0734 - out_env_loss: 0.7242 - out_sband_loss: 0.2810 - out_spec_loss: 0.2520 - val_loss: 0.8215 - val_out_bypass_loss: 0.0196 - val_out_env_loss: 0.6467 - val_out_sband_loss: 0.0922 - val_out_spec_loss: 0.1474 - learning_rate: 5.0000e-04
Epoch 2/100
2803/2804 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.8559 - out_bypass_loss: 0.0199 - out_env_loss: 0.6592 - out_sband_loss: 0.1490 - out_spec_loss: 0.1647
Epoch 2: val_loss improved from 0.82155 to 0.78207, saving model to /content/drive/MyDrive/best_join

In [ ]:
autoencoder.summary()

Model: "End_to_End_Latent_Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spec_input          │ (None, 32, 16, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bypass_input        │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ env_input           │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sub_band_input      │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Acoustic_Encoder_O… │ (None, 32)        │      9,664 │ spec_input[0][0], │
│ (Functional)        │                   │            │ bypass_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Vibration_Encoder_… │ (None, 32)        │     12,480 │ env_input[0][0],  │
│ (Functional)        │                   │            │ sub_band_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latent_fusion       │ (None, 64)        │          0 │ Acoustic_Encoder… │
│ (Concatenate)       │                   │            │ Vibration_Encode… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ latent_fusion[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ joint_bottleneck    │ (None, 2)         │        128 │ batch_normalizat… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2)         │          8 │ joint_bottleneck… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_85 (ReLU)     │ (None, 2)         │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_acoustic_sig    │ (None, 32)        │         96 │ re_lu_85[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_vibration_sig   │ (None, 32)        │         96 │ re_lu_85[0][0]    │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,634 (92.32 KB)

 Trainable params: 452 (1.77 KB)

 Non-trainable params: 22,276 (87.02 KB)

 Optimizer params: 906 (3.54 KB)

In [ ]:
import numpy as np

print("Extracting Balanced Micro-Decoder Test Errors...")

# 1. Find the file-level labels first so we know where they are
y_test = y_vib_test.reshape(1290, 40)[:, 0]

# 2. Grab the specific file indices for 300 Normal and 300 Anomaly files
normal_file_idx = np.where(y_test == 0)[0][:300]
anomaly_file_idx = np.where(y_test == 1)[0][:300]

# Combine them (600 files total)
balanced_file_indices = np.concatenate([normal_file_idx, anomaly_file_idx])

# 3. Convert those 600 file indices into the 24,000 individual frame indices
frame_indices = np.concatenate([np.arange(idx * 40, (idx + 1) * 40) for idx in balanced_file_indices])

# 4. Safely slice the arrays using our targeted indices
pred_spec, pred_bypass, pred_env, pred_sband = autoencoder.predict(
    {
        "spec_input": X_imfcc_test_scaled[frame_indices],
        "bypass_input": X_ae_test_scaled[frame_indices],
        "env_input": X_env_test_scaled[frame_indices],
        "sub_band_input": X_sband_test_scaled[frame_indices]
    },
    batch_size=32
)

# 5. Calculate the raw MSE per frame
err_spec = np.mean(np.square(X_imfcc_test_scaled[frame_indices] - pred_spec), axis=(1, 2, 3))
err_bypass = np.mean(np.square(X_ae_test_scaled[frame_indices] - pred_bypass), axis=1)
err_env = np.mean(np.square(X_env_test_scaled[frame_indices] - pred_env), axis=(1, 2))
err_sband = np.mean(np.square(X_sband_test_scaled[frame_indices] - pred_sband), axis=1)


# 6. Fuse the errors
total_proxy_error = err_bypass + err_sband + err_env + err_spec

# 7. Reshape and calculate final means
# The first 300 files are Normal, the next 300 are Anomaly
grouped_proxy_errors = total_proxy_error.reshape(600, 40)

raw_normal_proxy = np.mean(grouped_proxy_errors[:300])
raw_anomaly_proxy = np.mean(grouped_proxy_errors[300:])

print(f"RAW Normal Pipeline MSE:   {raw_normal_proxy:.5f}")
print(f"RAW Vandalism Impact MSE:  {raw_anomaly_proxy:.5f}")

Extracting Balanced Micro-Decoder Test Errors...
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step
RAW Normal Pipeline MSE:   0.67287
RAW Vandalism Impact MSE:  0.73798


In [ ]:
print("Extracting Micro-Decoder Test Errors...")

# Set your dynamic limit so you don't have to hunt down hardcoded numbers later
TEST_FILES = 600
FRAMES = TEST_FILES * 40

# 1. Run the sliced test set through the model
pred_spec, pred_bypass, pred_env, pred_sband = autoencoder.predict(
    {
        "spec_input": X_imfcc_test_scaled[:FRAMES],
        "bypass_input": X_ae_test_scaled[:FRAMES],
        "env_input": X_env_test_scaled[:FRAMES],
        "sub_band_input": X_sband_test_scaled[:FRAMES]
    },
    batch_size=32
)

# 2. Calculate the raw MSE per frame (Flattening all spatial dimensions)
# Spectrogram is 4D (batch, 32, 16, 1) -> average across axes 1, 2, and 3
err_spec = np.mean(np.square(X_imfcc_test_scaled[:FRAMES] - pred_spec), axis=(1, 2, 3))

# Bypass is 2D (batch, 6) -> average across axis 1
err_bypass = np.mean(np.square(X_ae_test_scaled[:FRAMES] - pred_bypass), axis=1)

# Envelope is 3D (batch, 100, 1) -> average across axes 1 and 2
err_env = np.mean(np.square(X_env_test_scaled[:FRAMES] - pred_env), axis=(1, 2))

# Sub-band is 2D (batch, 16) -> average across axis 1
err_sband = np.mean(np.square(X_sband_test_scaled[:FRAMES] - pred_sband), axis=1)

# 3. Fuse the errors (All four arrays are now perfectly flat: (24000,))
total_proxy_error = err_bypass + err_sband + err_env + err_spec

# 4. Separate and calculate the means based on ground truth labels
y_test_slashed = y_test[:FRAMES]
y_test_files = y_test_slashed.reshape(TEST_FILES, 40)[:, 0]

normal_files = np.where(y_test_files == 0)[0]
anomaly_files = np.where(y_test_files == 1)[0]

# 5. Reshape to the sliced 600 files, NOT 1290
grouped_proxy_errors = total_proxy_error.reshape(TEST_FILES, 40)

raw_normal_proxy = np.mean(grouped_proxy_errors[normal_files])
raw_anomaly_proxy = np.mean(grouped_proxy_errors[anomaly_files])

print(f"RAW Normal Pipeline MSE:   {raw_normal_proxy:.5f}")
print(f"RAW Vandalism Impact MSE:  {raw_anomaly_proxy:.5f}")

Extracting Micro-Decoder Test Errors...
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step
RAW Normal Pipeline MSE:   nan
RAW Vandalism Impact MSE:  0.87950


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
# @title
# Testing the Model on Training Data
print("Executing Joint Inference on Training Dataset")

# Extracting the Frozen Signatures for the Train Set
acoustic_train_signature = encoder_acoustic.predict(
    [X_imfcc_train_scaled, X_ae_train_scaled], batch_size=32
)

vibration_train_signature = encoder_vibration.predict(
    [X_env_train_scaled, X_sband_train_scaled], batch_size=32
)

# Reconstruction Prediction from the Autoencoder
train_recon_acoustic, train_recon_vibration = autoencoder.predict(
    {
        "spec_input" : X_imfcc_train_scaled,
        "bypass_input" : X_ae_train_scaled,
        "env_input" : X_env_train_scaled,
        "sub_band_input" : X_sband_train_scaled
    },
    batch_size=32
)

train_err_acoustic = np.mean(np.square(acoustic_train_signature - train_recon_acoustic), axis=1)
train_err_vibration = np.mean(np.square(vibration_train_signature - train_recon_vibration), axis=1)

train_err_acoustic.shape

Executing Joint Inference on Training Dataset
3505/3505 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step
3505/3505 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step
3505/3505 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step


(112160,)

In [ ]:
# @title
from sklearn.preprocessing import StandardScaler

scaler_ac = StandardScaler()
scaled_err_ac = scaler_ac.fit(train_err_acoustic.reshape(-1, 1))
scaler_vb = StandardScaler()
scaled_err_vb = scaler_vb.fit(train_err_vibration.reshape(-1, 1))

In [ ]:
# Testing the Model on Testing Data
print("Executing Joint Inference on Test Dataset")

# Extracting the Frozen Signatures for the Test Set
acoustic_test_signature = encoder_acoustic.predict(
    [X_imfcc_test_scaled, X_ae_test_scaled], batch_size=32
)

vibration_test_signature = encoder_vibration.predict(
    [X_env_test_scaled, X_sband_test_scaled], batch_size=32
)

Executing Joint Inference on Test Dataset
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


In [ ]:
# Reconstruction Prediction from the Autoencoder
test_recon_acoustic, test_recon_vibration = autoencoder.predict(
    {
        "spec_input" : X_imfcc_test_scaled,
        "bypass_input" : X_ae_test_scaled,
        "env_input" : X_env_test_scaled,
        "sub_band_input" : X_sband_test_scaled
    },
    batch_size=32
)

1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step


In [ ]:
test_err_acoustic = np.mean(np.square(acoustic_test_signature - test_recon_acoustic), axis=1)
test_err_vibration = np.mean(np.square(vibration_test_signature - test_recon_vibration), axis=1)

In [ ]:
test_err_ac_z = scaler_ac.transform(test_err_acoustic.reshape(-1, 1)).flatten()
test_err_vb_z = scaler_vb.transform(test_err_vibration.reshape(-1, 1)).flatten()

In [ ]:
clipped_z_ac = np.maximum(0, test_err_ac_z)
clipped_z_vb = np.maximum(0, test_err_vb_z)

# Final Joint Anomaly Score
test_recon_errors_joint = clipped_z_ac + clipped_z_vb

test_recon_errors_joint.shape

(51600,)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

z_thresholds = [1.0, 1.5, 2.0, 2.5]
voting_ratios = [0.10, 0.15, 0.20, 0.25]

num_test_files = 1290
frames_per_file = 40

errors_per_file = test_recon_errors_joint.reshape(num_test_files, frames_per_file)
if len(y_test) == (num_test_files * frames_per_file):
  label_per_file = y_test.reshape(num_test_files, frames_per_file)[:, 0]
else:
  print("The labels are not the expected shape as the errors")

for z_threshold in z_thresholds:
  for voting_ratio in voting_ratios:
    print(f"\n ================================")
    print(f"Evaluating: Frame Z-Threshold = {z_threshold} | Voting Ratio = {voting_ratio}")
    print(f"=================================")

    file_predictions = []

    for i in range(num_test_files):
      file_errors = errors_per_file[i]
      file_labels = labels_per_file

      anomalous_frame_count = np.sum(file_errors > z_threshold)

      percent_anomalous = anomalous_frame_count / frames_per_file

      # Apply Voting Threshold
      if percent_anomalous >= voting_ratio:
        predicted_label = 1
      else:
        predicted_label = 0

      file_predictions.append(predicted_label)

    # Print the exact metrics you used earlier
    print("--- Classification Report ---")
    print(classification_report(label_per_file, file_predictions, target_names=["Normal (0)", "Anomaly (1)"]))

    acc = accuracy_score(label_per_file, file_predictions) * 100
    print(f"Overall Accuracy: {acc:.2f}%\n")



Evaluating: Frame Z-Threshold = 1.0 | Voting Ratio = 0.1
--- Classification Report ---
              precision    recall  f1-score   support

  Normal (0)       0.30      0.66      0.41       400
 Anomaly (1)       0.66      0.30      0.41       890

    accuracy                           0.41      1290
   macro avg       0.48      0.48      0.41      1290
weighted avg       0.55      0.41      0.41      1290

Overall Accuracy: 40.93%


Evaluating: Frame Z-Threshold = 1.0 | Voting Ratio = 0.15
--- Classification Report ---
              precision    recall  f1-score   support

  Normal (0)       0.29      0.72      0.41       400
 Anomaly (1)       0.63      0.22      0.32       890

    accuracy                           0.37      1290
   macro avg       0.46      0.47      0.37      1290
weighted avg       0.52      0.37      0.35      1290

Overall Accuracy: 37.13%


Evaluating: Frame Z-Threshold = 1.0 | Voting Ratio = 0.2
--- Classification Report ---
              precision    re

In [ ]:
# 1. Grab all Normal file indices (where actual label == 0)
normal_files = np.where(y_test == 0)[0]
# 2. Grab all Anomaly file indices (where actual label == 1)
anomaly_files = np.where(y_test == 1)[0]

# 3. Calculate the RAW mean error across all frames for normal vs anomaly
raw_normal_error = np.mean(test_err_acoustic[normal_files] + test_err_vibration[normal_files])
raw_anomaly_error = np.mean(test_err_acoustic[anomaly_files] + test_err_vibration[anomaly_files])

print(f"RAW Normal Pipeline MSE:   {raw_normal_error:.5f}")
print(f"RAW Vandalism Impact MSE:  {raw_anomaly_error:.5f}")

RAW Normal Pipeline MSE:   0.14112
RAW Vandalism Impact MSE:  0.14563


In [ ]:
X_imfcc_train_scaled,
    X_ae_train_scaled,
    X_env_train_scaled,
    X_sband_train_scaled,